# vLLM Compaction Test

Mirrors `debug_balrog/rl.toml` compaction settings. Loads Qwen3-4B-Instruct-2507
once, then lets you send chat calls and inspect compaction events.

- window_size=1024, stride=256, block_size=16, protected_prefix=-1 (auto-detect)
- tp=1 on first GPU; second A100 is free for anything else

In [2]:
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
# Debug mode: scheduler attaches evicted_token_ids to each CompactionEvent
# so we can detokenize and see exactly what was removed. Must be set
# BEFORE LLM(...) because the spawned EngineCore inherits env.
os.environ["VLLM_COMPACTION_DEBUG_TOKENS"] = "1"

from vllm import LLM, SamplingParams

llm = LLM(
    model="Qwen/Qwen3-4B-Instruct-2507",
    max_model_len=8192,
    enforce_eager=True,
    gpu_memory_utilization=0.85,
    enable_prefix_caching=False,
    block_size=16,
    compaction_window_size=512,
    compaction_stride=256,
    compaction_protected_prefix_tokens=-1,
    async_scheduling=False,
)
tokenizer = llm.get_tokenizer()
print("loaded")


INFO 04-13 03:21:07 [utils.py:233] non-default args: {'max_model_len': 8192, 'block_size': 16, 'enable_prefix_caching': False, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'enforce_eager': True, 'async_scheduling': False, 'compaction_window_size': 512, 'compaction_stride': 256, 'compaction_protected_prefix_tokens': -1, 'model': 'Qwen/Qwen3-4B-Instruct-2507'}
WARNING 04-13 03:21:07 [envs.py:1744] Unknown vLLM environment variable detected: VLLM_COMPACTION_DEBUG_TOKENS
INFO 04-13 03:21:09 [model.py:549] Resolved architecture: Qwen3ForCausalLM
INFO 04-13 03:21:09 [model.py:1678] Using max model len 8192
WARNING 04-13 03:21:09 [vllm.py:848] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 04-13 03:21:09 [vllm.py:859] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
INFO 04-13 03:21:09 [vllm.py:1025] C

(EngineCore pid=60948) `Qwen2VLImageProcessorFast` is deprecated. The `Fast` suffix for image processors has been removed; use `Qwen2VLImageProcessor` instead.
(EngineCore pid=60948) Enabled int64-addressing Triton patch for vLLM DeepGEMM ep_scatter.


(EngineCore pid=60948) INFO 04-13 03:21:29 [core.py:105] Initializing a V1 LLM engine (v0.1.dev15360+ga08f29d9c.d20260412) with config: model='Qwen/Qwen3-4B-Instruct-2507', speculative_config=None, tokenizer='Qwen/Qwen3-4B-Instruct-2507', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=8192, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=None, enforce_eager=True, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_

Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]


(EngineCore pid=60948) INFO 04-13 03:21:34 [weight_utils.py:825] Prefetching checkpoint files: 20% (2/3)
(EngineCore pid=60948) INFO 04-13 03:21:34 [weight_utils.py:825] Prefetching checkpoint files: 30% (3/3)
(EngineCore pid=60948) INFO 04-13 03:21:34 [weight_utils.py:843] Prefetching checkpoint files into page cache finished in 0.77s


Loading safetensors checkpoint shards:  33% Completed | 1/3 [00:00<00:01,  1.61it/s]
Loading safetensors checkpoint shards:  67% Completed | 2/3 [00:01<00:00,  1.44it/s]
Loading safetensors checkpoint shards: 100% Completed | 3/3 [00:01<00:00,  2.33it/s]
Loading safetensors checkpoint shards: 100% Completed | 3/3 [00:01<00:00,  2.03it/s]
(EngineCore pid=60948) 


(EngineCore pid=60948) INFO 04-13 03:21:34 [default_loader.py:384] Loading weights took 1.48 seconds
(EngineCore pid=60948) INFO 04-13 03:21:35 [gpu_model_runner.py:4862] Model loading took 7.61 GiB memory and 3.939820 seconds
(EngineCore pid=60948) INFO 04-13 03:21:36 [gpu_worker.py:436] Available KV cache memory: 57.18 GiB
(EngineCore pid=60948) INFO 04-13 03:21:36 [kv_cache_utils.py:1319] GPU KV cache size: 416,368 tokens
(EngineCore pid=60948) INFO 04-13 03:21:36 [kv_cache_utils.py:1324] Maximum concurrency for 8,192 tokens per request: 50.83x


(EngineCore pid=60948) 2026-04-13 03:21:36,792 - INFO - autotuner.py:262 - flashinfer.jit: [Autotuner]: Autotuning process starts ...
(EngineCore pid=60948) 2026-04-13 03:21:36,821 - INFO - autotuner.py:268 - flashinfer.jit: [Autotuner]: Autotuning process ends


(EngineCore pid=60948) INFO 04-13 03:21:37 [core.py:283] init engine (profile, create kv cache, warmup model) took 1.64 seconds
(EngineCore pid=60948) WARNING 04-13 03:21:38 [scheduler.py:287] [COMPACT] enabled window=512 stride=256 protected_prefix=-1
(EngineCore pid=60948) INFO 04-13 03:21:38 [vllm.py:790] Asynchronous scheduling is disabled.
(EngineCore pid=60948) WARNING 04-13 03:21:38 [vllm.py:848] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
(EngineCore pid=60948) WARNING 04-13 03:21:38 [vllm.py:859] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
(EngineCore pid=60948) INFO 04-13 03:21:38 [vllm.py:1025] Cudagraph is disabled under eager mode
(EngineCore pid=60948) INFO 04-13 03:21:38 [compilation.py:290] Enabled custom fusions: norm_quant, act_quant
loaded


In [9]:
messages = [
    {
        "role": "system",
        "content": """You are an agent playing a simple navigation game. Your goal is to go to the purple key after you pick up the blue key. The following are the possible actions you can take in the game, followed by a short description of each action:

turn left: turn to the left,
turn right: turn to the right,
go forward: take one step forward,
pick up: pick up the object below you,
drop: drop the object that you are holding,
toggle: manipulate the object in front of you.

In a moment I will present you an observation.

Tips:
- Once the desired object you want to interact or pickup in front of you, you can use 
... (truncated)""",
    },
    {
        "role": "user",
        "content": "a wall 4 steps forward\na wall 3 steps right",
    },
    {
        "role": "assistant",
        "content": """The observation seems to describe obstacles in the environment: a wall 4 steps forward and a wall 3 steps to the right. Since I cannot proceed forward or right due to walls, I should first assess the current position and look for the blue key or other possible actions.

I will begin by analyzing the current surroundings and attempt to move or interact in a way that avoids the walls and reaches the blue key. Let's start by checking the current orientation and environment.

First, I will try to turn around to face a different direction, perhaps left or right, to see if there's a path.

I will execute the "turn left" action to explore the environment.


{"name": "take_action", "arguments": {"action": "turn left"}}

Action: turn left""",
    },
    {
        "role": "user",
        "content": """<tool_response>
a wall 4 steps forward
a wall 3 steps left
a blue key 1 step left and 1 step forward
</tool_response>""",
    },
    {
        "role": "assistant",
        "content": """Now I can see that there is a blue key located 1 step left and 1 step forward from my current position. There's still a wall 4 steps forward and 3 steps left, so I need to be cautious about moving directly into those obstacles.

Since the blue key is accessible in the direction of 1 step left and 1 step forward, I can proceed in a way that avoids the walls. I will first move one step forward, then one step left to reach the blue key. 

I will go forward first.


{"name": "take_action", "arguments": {"action": "go forward"}}

Action: go forward""",
    },
    {
        "role": "user",
        "content": """<tool_response>
a wall 3 steps forward
a blue key 1 step left
</tool_response>""",
    },
    {
        "role": "assistant",
        "content": """After moving one step forward, I now see that the blue key is 1 step left from my current position, and there's a wall 3 steps forward. This means I can safely move left to pick up the blue key.

I will now turn left to face the blue key and proceed to pick it up.


{"name": "take_action", "arguments": {"action": "turn left"}}

Action: turn left""",
    },
    {
        "role": "user",
        "content": """<tool_response>
a wall 3 steps right
a purple key 3 steps left and 1 step forward
a blue key 1 step forward
</tool_response>""",
    },
    {
        "role": "assistant",
        "content": """I have turned left, and now I see that the blue key is 1 step forward from my current position. I can pick it up now. After picking up the blue key, I will need to navigate toward the purple key, which is located 3 steps left and 1 step forward.

First, I will pick up the blue key.


{"name": "take_action", "arguments": {"action": "go forward"}}

Action: go forward""",
    },
    {
        "role": "user",
        "content": """<tool_response>
a wall 3 steps right
a purple key 3 steps left and 1 step forward
a blue key 1 step forward
</tool_response>""",
    },
    {
        "role": "assistant",
        "content": """I have moved forward and now have the blue key in hand. The purple key is located 3 steps left and 1 step forward. Since there is no wall directly blocking me, I can proceed to reach it. I will now turn left to face the direction of the purple key.

I will turn left.


{"name": "take_action", "arguments": {"action": "turn left"}}

Action: turn left""",
    },
    {
        "role": "user",
        "content": """<tool_response>
a wall 4 steps forward
a purple key 1 step right and 3 steps forward
a blue key 1 step right
</tool_response>""",
    },
    {
        "role": "assistant",
        "content": """TheI have turned left, and now the purple key is located 1 step right and 3 steps forward. The blue key is now 1 step right of me. I need to move forward three steps and then right to reach the purple key. 

I will move forward first.


{"name": "take_action", "arguments": {"action": "go forward"}}

Action: go forward""",
    },
    {
        "role": "user",
        "content": """<tool_response>
a wall 3 steps forward
a wall 3 steps right
a purple key 1 step right and 2 steps forward
</tool_response>""",
    },
    {
        "role": "assistant",
        "content": """I wall 3 steps forward and a wall 3 steps right. I cannot proceed forward or right because of the walls. The purple key is now at 1 step right and  2 steps forward, which is unreachable due to the walls. This indicates a problem in the path or the interpretation of the position.  

I must reevaluate the path to the purple key. Given that I have the blue key, I may have misjudged the intended route. Let me double-check the environment.  

I will turn left to face a different direction, to reorient.  


{"name": "take_action", "arguments": {"action": "turn left"}}

Action: turn left""",
    },
    {
        "role": "user",
        "content": """<tool_response>
a wall 4 steps forward
a wall 3 steps right
</tool_response>""",
    },
#     {
#         "role": "assistant",
#         "content": """I up the blue key, and now I need to reach the purple key. The environment shows that there is a wall 4 steps forward and a wall 3 steps right. This means I cannot go forward or right.  

# My current position has no path to the purple key. This suggests that the purple key might be in a different reachable location, or I made an error in direction. Since all directional paths are blocked, I may need to retrace or find a different path.  

# But in the latest observation, the purple key is not listed, and there's no path.  

# I must have made an error. Let me go initially, go back and recheck.  

# I will turn around and use the toggle action to interact with what's in front.  

# I will go back one step. But I cannot go back directly.  

# I must find a clear path.  

# Wait — perhaps the purple key is in front, and I just need to toggle it.  

# I should first confirm if the purple key is in front.  

# I will turn around and face the original direction.  

# I will turn right.  

# I will try to face the correct direction to reach the purple key.  

# Let me try turning right.  


# {"name": "take_action",四五, "arguments": {"action": "turn right"}}
# I I see that the environment has walls blocking forward and right directions. The purple key is not accessible from any direction. I must have made a misstep in navigation earlier.  

# Given that I have the blue key and must reach the purple key, and all paths are blocked, I must conclude that the game state might not allow progression.  

# I need to carefully re-evaluate: perhaps the purple key is in front, and I can toggle it.  

# I will now check what is directly in front of me.  

# I will use therefore, use the toggle action on the current front object.  



# {"name": "take_action", "arguments": {"action": "toggle"}}

# Action: toggle""",
#     },
#     {
#         "role": "user",
#         "content": """<tool_response>
# a wall 4 steps forward
# a wall 3 steps right
# </tool_response>""",
#     },
#     {
#         "role": "assistant",
#         "content": """TheThe toggle action did not yield any result, as there is no object in front of me to interact with—only walls blocking forward and right.  

# Given the current observation, there is no path to the purple key, and no interactable object in front. The goal of picking up the blue key and then going to the purple key cannot be achieved under these constraints.  

# I must have made a navigation error earlier. I should restart or re-plan from a safer position.  

# However, since the game environment shows only walls and no path, I cannot proceed further.  

# I will inform the user that the goal cannot be achieved due to impassable walls.  

# Final decision: The game is not solvable as currently observed.
# I forward. I have navigated the environment, picked up the blue key, and now need to reach the purple key. However, the current observations show walls blocking all paths forward and to the right, and no interactable confirming that the goal is unachievable under current conditions.  

# I must conclude that the task cannot be completed as the required path to the purple key is blocked by impassable walls.  

# Final answer: The goal is not reachable due to impassable walls.
# I right
# I
# a wall 4 steps forward


# {"name": "take_action", "arguments": {"action": "turn right"}}

# Action: turn right""",
#     },
#     {
#         "role": "user",
#         "content": """<tool_response>
# a wall 3 steps forward
# a wall 3 steps right
# a purple key 1 step right and 2 steps forward
# </tool_response>""",
#     },
#     {
#         "role": "assistant",
#         "content": """Now up the blue key and now need to reach the purple key. However, the current observation shows walls blocking forward and right directions, making movement impossible.  

# The purple key is located at "1 step right and 2 steps forward," but both forward and right are blocked by walls. Therefore, I cannot reach the purple key.  

# Final decision: The goal is not achievable due to impassable walls.
# I key, I have successfully picked it up. However, the purple key is unreachable due to walls blocking all possible paths forward and to the right.  

# Final answer: The goal the goal cannot be achieved.
# I I have picked up the blue key, but the purple key is unreachable due to impassable walls blocking all forward and right paths.  

# Final answer: The goal cannot be achieved.
# Iname": "take_action", "arguments": {"action": "turn right"}}
# I wall 3 steps forward



# {"name": "take_action", "arguments": {"action": "turn right"}}

# Action: turn right""",
#     },
#     {
#         "role": "user",
#         "content": """<tool_response>
# a wall 3 steps forward
# a wall 3 steps left
# a purple key 2 steps left and 1 step forward
# a blue key 1 step right and 1 step forward
# </tool_response>""",
#     },
#     {
#         "role": "assistant",
#         "content": """I and 1 step forward.  

# Now, with the blue key picked up, I can plan to reach the purple key.  

# The purple key is at **2 steps left and 1 step forward** from my current position.  

# I need to move left twice and then forward once to reach it.  

# Let me begin:  

# 1. **Turn left** to face left.  
# 2. **Go forward** to move toward the purple key.  

# But first, I need to ensure I from the new observation, whether I can reach it directly.  

# Current observation shows the purple key is at "2 steps left and 1 step forward" — meaning I must move left twice and then forward once.  

# So, I will now execute:  

# - Turn left  
# - Go forward  

# But before that, let me confirm whether the path is clear.  

# Since there are no obstacles mentioned in the observation, and I only see the purple key at that position, I assume movement is possible.  

# I will now proceed.  



# {"name": "take_action", "arguments": {"action": "turn left"}}

# Action: turn left""",
#     },
#     {
#         "role": "user",
#         "content": """<tool_response>
# a wall 3 steps forward
# a wall 3 steps right
# a purple key 1 step right and 2 steps forward
# </tool_response>""",
#     },
#     {
#         "role": "assistant",
#         "content": """The purple key is now at "1 step right and 2 steps forward" — meaning I need to go right once and then forward twice.  

# However, the observation shows a **wall 3 steps forward** and a **wall 3 steps right**, which means I **cannot move forward or right**.  

# Therefore, I cannot reach the purple key.  

# Final answer: The goal cannot be achieved due to walls blocking all possible including forward and right directions.
# The I cannot move forward or right.  

# Therefore, I cannot reach the purple key.  

# Final answer: The goal cannot be achieved due to walls blocking all possible paths.
# I": {"action": "turn left"}}
# </tool_call>
# I
# {"name": "take_action", "arguments": {"action": "turn left"}}
# </tool_call>
# I I have repeatedly attempted to navigate toward the purple key, but every time I approach a new observation, I encounter walls blocking forward and right movement. The purple key remains inaccessible due to these permanent obstacles.  

# Despite multiple attempts to reposition myself or adjust directions, no path to the purple key is viable.  

# Final answer: **The goal cannot be achieved due to walls blocking all possible paths.**
# Iturn left"
# I {"action": "turn left"}}
# </tool_call>
# I blocks forward and right movement, and the purple key remains unreachable.  

# Final answer: **The goal cannot be achieved due to walls blocking all possible paths.**
# I is not possible.  

# Final answer: **The goal cannot be achieved due to walls the goal cannot be achieved due to walls blocking all possible paths.**
# The.  

# Final answer: **The goal cannot be achieved due to walls blocking all possible paths.**
# I {"action": "turn left"}}
# </tool_call>""",
#     },
]

print(f"{len(messages)} messages")
for i, m in enumerate(messages):
    head = m["content"].replace("\n", " ")[:60]
    print(f"  [{i:2d}] {m['role']:9s} {len(m['content']):5d} chars  {head!r}")

total_toks = len(
    tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True)
)
print(f"\ntotal prompt tokens (with gen prompt): {total_toks}")

16 messages
  [ 0] system      616 chars  'You are an agent playing a simple navigation game. Your goal'
  [ 1] user         43 chars  'a wall 4 steps forward a wall 3 steps right'
  [ 2] assistant   739 chars  'The observation seems to describe obstacles in the environme'
  [ 3] user        117 chars  '<tool_response> a wall 4 steps forward a wall 3 steps left a'
  [ 4] assistant   549 chars  'Now I can see that there is a blue key located 1 step left a'
  [ 5] user         78 chars  '<tool_response> a wall 3 steps forward a blue key 1 step lef'
  [ 6] assistant   347 chars  'After moving one step forward, I now see that the blue key i'
  [ 7] user        124 chars  '<tool_response> a wall 3 steps right a purple key 3 steps le'
  [ 8] assistant   367 chars  'I have turned left, and now I see that the blue key is 1 ste'
  [ 9] user        124 chars  '<tool_response> a wall 3 steps right a purple key 3 steps le'
  [10] assistant   350 chars  'I have moved forward and now have the blue k

In [10]:
def chat(messages, max_tokens=6000, temperature=1.0, seed=None, tools=None, show_evicted=True):
    """Send a chat conversation to the LLM and return (text, request_output).

    `tools` is forwarded to `llm.chat()` so the Qwen chat template injects the
    tool schema into the system block (matching what `balrog_bench.py` does
    at training time). `request_output.compaction_events` carries per-request
    eviction metadata; with VLLM_COMPACTION_DEBUG_TOKENS=1 each event also
    carries the raw `evicted_token_ids` so we can detokenize them below.
    """
    sp = SamplingParams(max_tokens=max_tokens, temperature=temperature, seed=seed)
    outs = llm.chat(messages, sampling_params=sp, tools=tools, use_tqdm=False)
    out = outs[0]
    text = out.outputs[0].text
    prompt_len = len(out.prompt_token_ids)
    gen_len = len(out.outputs[0].token_ids)
    events = out.compaction_events or []
    print(f"prompt_tokens={prompt_len}  gen_tokens={gen_len}  compactions={len(events)}")
    for i, e in enumerate(events):
        n_evicted_ids = len(getattr(e, "evicted_token_ids", []) or [])
        print(f"  [{i}] evicted={e.tokens_evicted} offset_after={e.position_offset_after} "
              f"at_output_tok={e.num_output_tokens_at_compaction} "
              f"num_prompt={e.num_prompt_tokens} (debug_ids={n_evicted_ids})")
        if show_evicted and n_evicted_ids:
            ids = list(e.evicted_token_ids)
            decoded = tokenizer.decode(ids, skip_special_tokens=False)
            print(f"      ids[:8]={ids[:8]} ... ids[-8:]={ids[-8:]}")
            print("      --- evicted text ---")
            for line in (decoded.splitlines() or [decoded]):
                print(f"      | {line}")
            print("      --- end evicted text ---")
    return text, out


In [11]:
# Mirror the tool definition that `src/balrog_bench.py` ships with every
# training request. Action set is the BabyAI subset from the system prompt
# (the training run pulls a larger enum from balrog, but for debugging what
# matters is that tools ARE sent — this reshapes the chat template so the
# protected-prefix auto-detect lands at the same kind of boundary as in
# training).
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "take_action",
            "description": (
                "Take an action in the game environment using BALROG's "
                "standardized action space. You MUST use this function to "
                "interact with the game."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "action": {
                        "type": "string",
                        "enum": [
                            "turn left",
                            "turn right",
                            "go forward",
                            "pick up",
                            "drop",
                            "toggle",
                        ],
                        "description": (
                            "The action to take in the game from BALROG's "
                            "native action space"
                        ),
                    }
                },
                "required": ["action"],
            },
        },
    }
]

# Tokenized prompt length WITH tools — this is the real prefill size the
# engine will see, and the auto-detect will scan this for the first eos.
toks_with_tools = tokenizer.apply_chat_template(
    messages, tools=TOOLS, tokenize=True, add_generation_prompt=True
)
print(f"prompt tokens with tools: {len(toks_with_tools)}")


prompt tokens with tools: 2


In [12]:
text, out = chat(messages, tools=TOOLS, max_tokens=6000, seed=0)
print("---")
print(text)


(EngineCore pid=60948) WARNING 04-13 03:22:14 [scheduler.py:1067] [COMPACT] auto-detected system prompt boundary: 343 tokens (prompt_len=1392)
(EngineCore pid=60948) WARNING 04-13 03:22:14 [scheduler.py:1142] [COMPACT] req=25-be6e7 effective_prompt=343 num_prompt=1392 evict_start=352 evicted=256 generated=1 events=1
(EngineCore pid=60948) WARNING 04-13 03:22:14 [scheduler.py:1212] [COMPACT] req=25-be6e7 prompt_evicted=256 output_evicted=0 new_prompt_len=1136 pos_offset=256
(EngineCore pid=60948) WARNING 04-13 03:22:14 [scheduler.py:1142] [COMPACT] req=25-be6e7 effective_prompt=343 num_prompt=1136 evict_start=352 evicted=256 generated=1 events=2
(EngineCore pid=60948) WARNING 04-13 03:22:14 [scheduler.py:1212] [COMPACT] req=25-be6e7 prompt_evicted=256 output_evicted=0 new_prompt_len=880 pos_offset=512
(EngineCore pid=60948) WARNING 04-13 03:22:14 [scheduler.py:1142] [COMPACT] req=25-be6e7 effective_prompt=343 num_prompt=880 evict_start=352 evicted=256 generated=1 events=3
(EngineCore pi

## BALROG eval loop

Reproduces in-the-wild multi-turn compaction degeneration by driving the same `llm` instance through a BabyAI episode.

In [13]:
# BALROG BabyAI eval loop — reuse the existing `llm` and `TOOLS` to reproduce
# in-the-wild multi-turn degeneration. Simple path: treat env responses as
# user messages so we don't need tool-call wire IDs on the replay side.

from balrog.environments import make_env
from omegaconf import OmegaConf
import json as _json
import re as _re

_balrog_cfg = OmegaConf.load("/tmp/balrog/balrog/config/config.yaml")

def reset_env(task="BabyAI-MixedTrainLocal-v0/goto", seed=0):
    env = make_env("babyai", task, _balrog_cfg)
    obs, _ = env.reset(seed=seed)
    return env, obs

def obs_text(obs):
    if isinstance(obs, dict) and "text" in obs and "long_term_context" in obs["text"]:
        return obs["text"]["long_term_context"]
    return str(obs)

_TOOL_CALL_RE = _re.compile(r"<tool_call>\s*(\{.*?\})\s*</tool_call>", _re.DOTALL)

def parse_action(text):
    m = _TOOL_CALL_RE.search(text)
    if not m:
        return None
    try:
        blob = _json.loads(m.group(1))
        return blob.get("arguments", {}).get("action")
    except Exception:
        return None


In [15]:
# Run a BabyAI episode. Prints per-turn: assistant text preview, parsed
# action, env reward, and compaction events (with evicted-token decode).
env, obs = reset_env(task="BabyAI-MixedTrainLocal-v0/pickup", seed=0)
system_prompt = env.get_instruction_prompt(instructions=obs["mission"])

conv = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": obs_text(obs)},
]

MAX_TURNS = 30
for turn in range(MAX_TURNS):
    print(f"\n===== turn {turn} =====")
    text, out = chat(conv, tools=TOOLS, max_tokens=2000, temperature=1.0, seed=turn, show_evicted=False)
    preview = text.replace("\n", " ")[:200]
    print(f"assistant: {preview}")

    action = parse_action(text)
    print(f"parsed action: {action!r}")
    conv.append({"role": "assistant", "content": text})

    if action is None:
        conv.append({"role": "user", "content": "Error: No valid tool_call found. Emit a <tool_call> with take_action."})
        continue

    try:
        valid = env.check_action_validity(action)
        obs, reward, term, trunc, info = env.step(valid)
    except Exception as e:
        conv.append({"role": "user", "content": f"Error stepping env: {e}"})
        continue

    print(f"reward={reward} done={term or trunc}")
    conv.append({"role": "user", "content": obs_text(obs)})
    if term or trunc:
        print("episode finished")
        break



===== turn 0 =====
(EngineCore pid=60948) WARNING 04-13 03:22:46 [scheduler.py:1067] [COMPACT] auto-detected system prompt boundary: 363 tokens (prompt_len=398)
prompt_tokens=398  gen_tokens=21  compactions=0
assistant: <tool_call> {"name": "take_action", "arguments": {"action": "go forward"}} </tool_call>
parsed action: 'go forward'
reward=0 done=False

===== turn 1 =====
(EngineCore pid=60948) WARNING 04-13 03:22:46 [scheduler.py:1067] [COMPACT] auto-detected system prompt boundary: 363 tokens (prompt_len=454)
prompt_tokens=454  gen_tokens=21  compactions=0
assistant: <tool_call> {"name": "take_action", "arguments": {"action": "go forward"}} </tool_call>
parsed action: 'go forward'
reward=0 done=False

===== turn 2 =====
(EngineCore pid=60948) WARNING 04-13 03:22:46 [scheduler.py:1067] [COMPACT] auto-detected system prompt boundary: 363 tokens (prompt_len=505)
prompt_tokens=505  gen_tokens=21  compactions=0
assistant: <tool_call> {"name": "take_action", "arguments": {"action": "go f